In [1]:
#Phase 1 — squelette (séquences + structure génique) : ~100 % fait
#Protéome, CDS, ARN, GFF pour maïs (B73), riz (IRGSP), tabac (K326 + TN90). C’est la couche la plus facile et la plus complète — les 4 assemblages de référence officiels sont là, avec checksums vérifiés. Rien de majeur ne manque ici.


In [ ]:
import os, shutil
from google.colab import drive
drive.mount("/content/drive")
ROOT = "/content/drive/MyDrive"
DRIVE_DIR = ROOT + "/files"
DATA_DIR = DRIVE_DIR + "/data"

for name in ("sources.yaml", "fetch_raw.py"):
    assert os.path.exists(f"{DRIVE_DIR}/{name}"), f"{name} absent de {DRIVE_DIR}"
    shutil.copy(f"{DRIVE_DIR}/{name}", ".")
print("Scripts copies depuis", DRIVE_DIR, "| donnees dans", DATA_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Scripts copies depuis /content/drive/MyDrive/files | donnees dans /content/drive/MyDrive/files/data


In [ ]:
import pandas as pd
OLD = f"{DRIVE_DIR}/data_from_colab"
if os.path.exists(OLD) and not os.path.exists(DATA_DIR):
    shutil.move(OLD, DATA_DIR)
    os.rename(f"{DATA_DIR}/manifest_colab.csv", f"{DATA_DIR}/manifest.csv")
    m = pd.read_csv(f"{DATA_DIR}/manifest.csv")
    m["file"] = m["file"].str.replace(r"^data/", "", regex=True)
    m.to_csv(f"{DATA_DIR}/manifest.csv", index=False)
    print("Migre vers", DATA_DIR, "-", len(m), "lignes de manifeste")
else:
    print("Rien a migrer (data existe deja ou data_from_colab absent).")

Rien a migrer (data existe deja ou data_from_colab absent).


In [ ]:
# Define directories and options
ROOT = "/content/drive/MyDrive"
DRIVE_DIR = ROOT + "/files"
DATA_DIR = DRIVE_DIR + "/data"
OPTIONS = ""

SOURCES = ["ncbi_refseq_zea_b73_v5", "ncbi_refseq_oryza_irgsp"]

import subprocess, shlex, sys
for sid in SOURCES:
    cmd = (f'python -u fetch_raw.py sources.yaml --source {sid} '
           f'--raw-dir "{DATA_DIR}/raw" --manifest "{DATA_DIR}/manifest.csv" --no-disk-check {OPTIONS}')
    print("$", cmd, flush=True)
    p = subprocess.Popen(shlex.split(cmd), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for ch in iter(lambda: p.stdout.read(1), ""):
        sys.stdout.write(ch); sys.stdout.flush()
    print("code de sortie :", p.wait(), "\n")

$ python -u fetch_raw.py sources.yaml --source ncbi_refseq_zea_b73_v5 --raw-dir "/content/drive/MyDrive/files/data/raw" --manifest "/content/drive/MyDrive/files/data/manifest.csv" --no-disk-check 

[ncbi_refseq_zea_b73_v5] 6 fichiers, ~86 Mo declares
    - GCF_902167145.1_Zm-B73-REFERENCE-NAM-5.0_protein.faa.gz  ~12 Mo
    - GCF_902167145.1_Zm-B73-REFERENCE-NAM-5.0_cds_from_genomic.fna.gz  ~18 Mo
    - GCF_902167145.1_Zm-B73-REFERENCE-NAM-5.0_rna.fna.gz  ~36 Mo
    - GCF_902167145.1_Zm-B73-REFERENCE-NAM-5.0_genomic.gff.gz  ~15 Mo
    - GCF_902167145.1_Zm-B73-REFERENCE-NAM-5.0_feature_table.txt.gz  ~4.3 Mo
    - GCF_902167145.1-RS_2025_02_gene_ontology.gaf.gz  ~1.1 Mo
  = GCF_902167145.1_Zm-B73-REFERENCE-NAM-5.0_protein.faa.gz deja present (non touche)
  = GCF_902167145.1_Zm-B73-REFERENCE-NAM-5.0_cds_from_genomic.fna.gz deja present (non touche)
  = GCF_902167145.1_Zm-B73-REFERENCE-NAM-5.0_rna.fna.gz deja present (non touche)
  = GCF_902167145.1_Zm-B73-REFERENCE-NAM-5.0_genomic.gff.gz d

In [ ]:
m = pd.read_csv(f"{DATA_DIR}/manifest.csv")
m["md5_ok"] = m["checksum_verified"].eq("oui")
summary = m.groupby("source_id").agg(fichiers=("file", "count"),
                                      Mo=("bytes", lambda b: round(b.sum() / 1e6, 1)),
                                      md5_ok=("md5_ok", "sum"))
display(summary)

,fichiers,Mo,md5_ok
source_id,,,
ncbi_refseq_nitab_asm71507v2,12,179.0,12
ncbi_refseq_nitab_tn90,7,159.1,7
ncbi_refseq_oryza_irgsp,6,66.3,6
ncbi_refseq_zea_b73_v5,6,90.8,6
ncbi_refseq_zea_b73_v5_extra,3,1.3,3
uniprot_proteomes_oryza_sativa,1,1.3,0
uniprot_proteomes_tabacum,1,0.1,0
uniprot_proteomes_zea_mays,1,0.2,0


In [ ]:
shutil.copy(f"{DRIVE_DIR}/write_phase_readme.py", ".")
!python write_phase_readme.py sources.yaml "{DATA_DIR}/manifest.csv" --phase 1 --out "{DATA_DIR}/README_phase1.md"

-> /content/drive/MyDrive/files/data/README_phase1.md (31 fichiers, 495.2 Mo, phase 1)
